In [1]:
import numpy as np
import pandas as pd 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv('train.csv')

In [4]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [5]:
df.shape

(891, 12)

In [6]:
# we have 891 rows

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [8]:
# we can see some values are missing

In [9]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

# Let's plan our pipeline

1. remove useless col if there.
2. Impute missing values. ---> age , embarked
3. transform -- one hot encoding --> sex , embarked
3. Scaling of all num col.
4. Feature selection --> top 5 col
5. model train --> dtc model

for all these we will use column transformer in each step and the output of one column transformer will be input of next column transformer

In [10]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [11]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [12]:
df = df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

In [13]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')

In [14]:
# for our ease 

df.rename(columns={'Survived':'survived',
                          'Pclass':'pclass',
                          'Sex':'sex',
                          'Age':'age',
                          'SibSp':'sibsp',
                          'Parch':'parch',
                          'Fare':'fare',
                          'Embarked':'embarked'})

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [15]:
data = df.rename(columns={'Survived':'survived',
                          'Pclass':'pclass',
                          'Sex':'sex',
                          'Age':'age',
                          'SibSp':'sibsp',
                          'Parch':'parch',
                          'Fare':'fare',
                          'Embarked':'embarked'})

In [16]:
data.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked'],
      dtype='object')

In [17]:
# now Train test split

x_train, x_test, y_train, y_test = train_test_split(data.drop(columns=['survived']),
                                                   data['survived'],
                                                   random_state=42,
                                                   test_size=0.2)

In [18]:
x_train.head()

,pclass,sex,age,sibsp,parch,fare,embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [19]:
# step1 -- imputation transformer ([var_name, SimpleImputer(strategy), ['col_name']])-- age , embarked

ColumnTransformer([
    ('impute_age', SimpleImputer(),[2]),
    ('impute_embarked',SimpleImputer(strategy='most_frequent'), [6])
], remainder='passthrough')


# here we intentionally using col index number instead of col name , cause transformers output are => numpy array
# and numpy array donot have col name  , so the code will break when we call the col by name in next 
# transformer, so for safety we use col index value , code will not break

# remainder ='passthrough' -- otherwise rest columns would have dropped

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_embarked',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [20]:
trf1 = ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])
],remainder='passthrough')

In [21]:
# step2 -  transform -- one hot encoding --> sex , embarked

ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])
], remainder='passthrough')


# we are not using -- drop='first' -- beacuse we are using decision tree classifier

ColumnTransformer(remainder='passthrough',
                  transformers=[('ohe_sex_embarked',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [1, 6])])

In [22]:
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])
],remainder='passthrough')

In [23]:
# Step3 -- scaling

ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
], remainder='passthrough')

ColumnTransformer(remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(), slice(0, 10, None))])

In [24]:
# Step3 -- scaling

trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
])

OneHotEncoder alone (when you call fit_transform on a DataFrame column) does not “keep” the original categorical column — it just spits out the encoded dummy matrix.

But when you wrap it inside a ColumnTransformer(..., remainder='passthrough'), things change.



### You told the ColumnTransformer:

Apply OHE on [1,6] (sex, embarked).

remainder='passthrough' → keep every other column untouched.

In [25]:
# step4 -- feature selection

SelectKBest(score_func=chi2,k=8)

# this means we are using 8 best columns only

SelectKBest(k=8, score_func=<function chi2 at 0x12da34ea0>)

In [26]:
trf4 = SelectKBest(score_func=chi2,k=8)


In [27]:
# step5 -- model object creation

trf5 = DecisionTreeClassifier()

now all nodes for pipe line are ready , 
# Now lets make pipeline using these nodes 

In [28]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5)
])

# Pipeline Vs make_pipeline

Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)

In [29]:
# Alternate Syntax
# pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [30]:
pipe

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8, score_func=<function chi2 at 0x12da34ea0>)),
                ('trf5', DecisionTreeClassifier())])

In [31]:
# Display pipeline

from sklearn import set_config
set_config(display='diagram')

# if you are not able to see this pipeline diagram above , cause in newer version of python , these diagrams are 
# shown by default

1. if we are only making pipe line for pre-processing then we will use ==> 
    pipe.fit_transform(x_train,y_train)
    
2. if we make pipe along with model also , so we use ==>
    pipe.fit(x_train, y_train)

In [32]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8, score_func=<function chi2 at 0x12da34ea0>)),
                ('trf5', DecisionTreeClassifier())])

# Explore pipelines --> named_steps 

In [33]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'trf3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x12da34ea0>),
 'trf5': DecisionTreeClassifier()}

In [34]:
# here we can see that everything is stored as dictionary
# here now we can drill down to see inside each node

pipe.named_steps['trf1']

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_embarked',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [35]:
# lets find the mean calculated in simpleimpuder in trf1

pipe.named_steps['trf1'].transformers_

[('impute_age', SimpleImputer(), [2]),
 ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 3, 4, 5])]

In [36]:
# since this is a list we will take first element 'tuple here' out via indexing [index_value]

pipe.named_steps['trf1'].transformers_[0]

('impute_age', SimpleImputer(), [2])

In [37]:
pipe.named_steps['trf1'].transformers_[0][1]

SimpleImputer()

In [39]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_

array([29.49884615])

In [40]:
# now lets fetch the value in second imputer in trf1

pipe.named_steps['trf1'].transformers_

[('impute_age', SimpleImputer(), [2]),
 ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 3, 4, 5])]

In [41]:
pipe.named_steps['trf1'].transformers_[1]

('impute_embarked', SimpleImputer(strategy='most_frequent'), [6])

In [42]:
pipe.named_steps['trf1'].transformers_[1][1]

SimpleImputer(strategy='most_frequent')

In [43]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [44]:
# so this is the most frequent value


# now since the pipe is ready lets check its performance

In [46]:
pipe.predict(x_test)

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0])

In [47]:
y_pred = pipe.predict(x_test)

In [48]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.6256983240223464

# Cross Validation using Pipeline

In [49]:
# cross validation using cross_val_score

from sklearn.model_selection import cross_val_score

In [50]:
cross_val_score(pipe, x_train, y_train, cv=5, scoring='accuracy').mean()

0.6391214419383433

In [51]:
# this runs the pipe for 5 x_train and y_train for 5 times then take mean of accuracy score

# Hyper parameter tuning via Grid Search using pipelines

In [52]:
# here since we are usnig DTC -- it has depth parameter we will tune that

para = {
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [53]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, para, cv=5, scoring='accuracy')
grid.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x12da34ea0>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [54]:
grid.best_score_

0.6391214419383433

In [55]:
grid.best_params_

{'trf5__max_depth': 2}

# Now to use this in production -- Export pipeline

In [56]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))